In [1]:
# NORTHSTAR — Solace Browser: JavaScript Quality QA
# DNA: js_qa = forbidden_states(var+eval+catch_ignore) x xss(escapeHtml) x globals(pollution) x modules(structure)
# Template: solace-cli/notebooks/qa-templates/template-page-qa.ipynb
# Towers: T1(Masters) T6(Hackers) T16(AI Coders) T23(Web)
# Auth: 65537 | Papers: 46,47,49,50
#
# File-based probes — reads all JS files directly
# REAL assertions. No mocks. No stubs.

import json, re, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "solace-browser-javascript-qa"
NOTEBOOK_PATH = "notebooks/qa/02-javascript-qa.ipynb"
PROJECT = "solace-browser"
MIN_SCORE = 70

BROWSER = Path("/home/phuc/projects/solace-browser")
WEB = BROWSER / "web"
JS_DIR = WEB / "js"

# All JS files in the project
JS_FILES = sorted(JS_DIR.glob("*.js"))
JS_NAMES = [f.stem for f in JS_FILES]

results = []

def record(probe_id, name, passed, detail="", tower_ref=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status,
                    "detail": detail, "tower_ref": tower_ref})
    print(f"  {status}: {name}" + (f" -- {detail}" if detail and not passed else ""))

print(f"BOOTSTRAP: {len(JS_FILES)} JS files")
print(f"Files: {JS_NAMES}")

BOOTSTRAP: 16 JS files
Files: ['layout', 'onboarding', 'schedule-approvals', 'schedule-calendar', 'schedule-cloud', 'schedule-core', 'schedule-evidence', 'schedule-old', 'setup-wizard', 'solace', 'theme', 'yinyang-delight', 'yinyang-oauth3-confirm', 'yinyang-rail', 'yinyang-tutorial-v2', 'yinyang-tutorial']


In [2]:
# PROBE 1: No `var` declarations — all should be const/let (ES6+)
# Tower ref: T1 F2 (Kent Beck — modern JS), T16 F3 (AI Coders — clean code)
print("=" * 60)
print("PROBE 1: No var Declarations (const/let only)")
print("=" * 60)

for jsf in JS_FILES:
    src = jsf.read_text(encoding="utf-8")
    # Match `var ` at start of line or after semicolon/brace — real declarations
    # Exclude comments and strings (rough heuristic: lines starting with var)
    var_lines = []
    for i, line in enumerate(src.split("\n"), 1):
        stripped = line.strip()
        # Skip comments
        if stripped.startswith("//") or stripped.startswith("*") or stripped.startswith("/*"):
            continue
        # Match var declarations
        if re.search(r'\bvar\s+\w', stripped):
            var_lines.append((i, stripped[:80]))

    record(f"P1-no-var-{jsf.stem}", f"{jsf.stem}: no var declarations ({len(var_lines)} found)",
           len(var_lines) == 0,
           f"lines: {[v[0] for v in var_lines[:5]]}" if var_lines else "",
           "T1-F2")

PROBE 1: No var Declarations (const/let only)
  PASS: layout: no var declarations (0 found)
  PASS: onboarding: no var declarations (0 found)
  PASS: schedule-approvals: no var declarations (0 found)
  PASS: schedule-calendar: no var declarations (0 found)
  PASS: schedule-cloud: no var declarations (0 found)
  PASS: schedule-core: no var declarations (0 found)
  PASS: schedule-evidence: no var declarations (0 found)
  PASS: schedule-old: no var declarations (0 found)
  PASS: setup-wizard: no var declarations (0 found)
  PASS: solace: no var declarations (0 found)
  PASS: theme: no var declarations (0 found)
  PASS: yinyang-delight: no var declarations (0 found)
  PASS: yinyang-oauth3-confirm: no var declarations (0 found)
  PASS: yinyang-rail: no var declarations (0 found)
  PASS: yinyang-tutorial-v2: no var declarations (0 found)
  PASS: yinyang-tutorial: no var declarations (0 found)


In [3]:
# PROBE 2: No eval() or Function() — XSS vectors
# Tower ref: T6 F1 (Hackers — code injection), T1 F2 (Kent Beck — safe code)
print("=" * 60)
print("PROBE 2: No eval() / Function() / new Function()")
print("=" * 60)

for jsf in JS_FILES:
    src = jsf.read_text(encoding="utf-8")
    # Find eval( and new Function( outside comments
    eval_hits = []
    func_hits = []
    for i, line in enumerate(src.split("\n"), 1):
        stripped = line.strip()
        if stripped.startswith("//") or stripped.startswith("*"):
            continue
        if "eval(" in stripped:
            eval_hits.append(i)
        if "Function(" in stripped and "new Function" in stripped:
            func_hits.append(i)

    record(f"P2-no-eval-{jsf.stem}", f"{jsf.stem}: no eval() ({len(eval_hits)} found)",
           len(eval_hits) == 0,
           f"lines: {eval_hits}" if eval_hits else "",
           "T6-F1")
    record(f"P2-no-func-{jsf.stem}", f"{jsf.stem}: no new Function() ({len(func_hits)} found)",
           len(func_hits) == 0,
           f"lines: {func_hits}" if func_hits else "",
           "T6-F1")

PROBE 2: No eval() / Function() / new Function()
  PASS: layout: no eval() (0 found)
  PASS: layout: no new Function() (0 found)
  PASS: onboarding: no eval() (0 found)
  PASS: onboarding: no new Function() (0 found)
  PASS: schedule-approvals: no eval() (0 found)
  PASS: schedule-approvals: no new Function() (0 found)
  PASS: schedule-calendar: no eval() (0 found)
  PASS: schedule-calendar: no new Function() (0 found)
  PASS: schedule-cloud: no eval() (0 found)
  PASS: schedule-cloud: no new Function() (0 found)
  PASS: schedule-core: no eval() (0 found)
  PASS: schedule-core: no new Function() (0 found)
  PASS: schedule-evidence: no eval() (0 found)
  PASS: schedule-evidence: no new Function() (0 found)
  PASS: schedule-old: no eval() (0 found)
  PASS: schedule-old: no new Function() (0 found)
  PASS: setup-wizard: no eval() (0 found)
  PASS: setup-wizard: no new Function() (0 found)
  PASS: solace: no eval() (0 found)
  PASS: solace: no new Function() (0 found)
  PASS: theme: no eva

In [4]:
# PROBE 3: innerHTML uses escapeHtml — XSS protection
# Tower ref: T6 F1 (Hackers — XSS), T6 F8 (Security Headers)
# NOTE: innerHTML with static HTML templates is safe. Only user-data interpolation is risky.
# Safe patterns: string literals, template HTML, escapeHtml(), textContent
print("=" * 60)
print("PROBE 3: innerHTML Uses escapeHtml")
print("=" * 60)

for jsf in JS_FILES:
    src = jsf.read_text(encoding="utf-8")
    lines = src.split("\n")

    innerHTML_count = src.count(".innerHTML")
    # Check if the file defines or uses escapeHtml
    has_escape_fn = "escapeHtml" in src or "escape_html" in src
    # Check if it uses textContent for setting text (safe alternative)
    has_textcontent = "textContent" in src

    if innerHTML_count == 0:
        record(f"P3-xss-{jsf.stem}", f"{jsf.stem}: no innerHTML usage (safe)",
               True, tower_ref="T6-F1")
    elif has_escape_fn or has_textcontent:
        record(f"P3-xss-{jsf.stem}",
               f"{jsf.stem}: innerHTML ({innerHTML_count} uses) with escapeHtml/textContent available",
               True, f"{innerHTML_count} uses, escape function present", "T6-F1")
    else:
        # No escape function at all — this is the real concern
        record(f"P3-xss-{jsf.stem}",
               f"{jsf.stem}: innerHTML ({innerHTML_count} uses) WITHOUT escapeHtml",
               False,
               f"{innerHTML_count} innerHTML uses but no escapeHtml or textContent defined",
               "T6-F1")

PROBE 3: innerHTML Uses escapeHtml
  PASS: layout: innerHTML (3 uses) with escapeHtml/textContent available
  PASS: onboarding: innerHTML (3 uses) with escapeHtml/textContent available
  PASS: schedule-approvals: innerHTML (7 uses) with escapeHtml/textContent available
  PASS: schedule-calendar: innerHTML (6 uses) with escapeHtml/textContent available
  PASS: schedule-cloud: no innerHTML usage (safe)
  PASS: schedule-core: no innerHTML usage (safe)
  PASS: schedule-evidence: innerHTML (15 uses) with escapeHtml/textContent available
  PASS: schedule-old: innerHTML (27 uses) with escapeHtml/textContent available
  PASS: setup-wizard: innerHTML (2 uses) with escapeHtml/textContent available
  PASS: solace: innerHTML (34 uses) with escapeHtml/textContent available
  PASS: theme: no innerHTML usage (safe)
  PASS: yinyang-delight: no innerHTML usage (safe)
  PASS: yinyang-oauth3-confirm: innerHTML (1 uses) with escapeHtml/textContent available
  PASS: yinyang-rail: innerHTML (11 uses) with e

In [5]:
# PROBE 4: No catch-and-ignore (empty catch blocks without even a comment)
# Tower ref: T1 F2 (Kent Beck — fail loud), FALLBACK BAN
# Note: catches with comments (/* quota */, /* Safari */) or console.debug are acceptable
print("=" * 60)
print("PROBE 4: No Catch-and-Ignore (Fallback Ban)")
print("=" * 60)

for jsf in JS_FILES:
    src = jsf.read_text(encoding="utf-8")
    lines = src.split("\n")

    bad_catches = []
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        i += 1

        # Single-line try/catch pattern: try { x } catch (_) { }
        if re.search(r'catch\s*\([^)]*\)\s*\{\s*\}$', line):
            # Truly empty — no comment, no code
            bad_catches.append((i, "empty catch"))
            continue

        # Single-line catch with only a comment is acceptable: catch (_) { /* reason */ }
        if re.search(r'catch\s*\([^)]*\)\s*\{.*?/[/*].*\}', line):
            continue  # has comment — acceptable

        # Multi-line catch blocks
        match = re.search(r'catch\s*\([^)]*\)\s*\{', line)
        if match and '}' not in line[match.end():]:
            catch_start = i
            body_lines = []
            while i < len(lines):
                body_line = lines[i].strip()
                i += 1
                if '}' in body_line:
                    body_lines.append(body_line.replace('}', '').strip())
                    break
                body_lines.append(body_line)
            body = " ".join(body_lines).strip()
            # Empty body (no code, no comment)
            if not body:
                bad_catches.append((catch_start, "empty catch"))
            # Body is just return null/undefined/""/{}/[]
            elif re.match(r'^return\s+(null|undefined|""|\'\'|\[\]|\{\})\s*;?$', body):
                bad_catches.append((catch_start, "catch-return-null"))

    # .catch(function() {}) on promises — single-line empty
    promise_empty = re.findall(r'\.catch\(function\s*\([^)]*\)\s*\{\s*\}\)', src)

    record(f"P4-no-catch-ignore-{jsf.stem}",
           f"{jsf.stem}: no empty catches ({len(bad_catches)} try/catch, {len(promise_empty)} promise)",
           len(bad_catches) == 0 and len(promise_empty) == 0,
           f"try/catch: {[c[0] for c in bad_catches]}, promise: {len(promise_empty)}" if (bad_catches or promise_empty) else "",
           "T1-F2")

PROBE 4: No Catch-and-Ignore (Fallback Ban)
  PASS: layout: no empty catches (0 try/catch, 0 promise)
  PASS: onboarding: no empty catches (0 try/catch, 0 promise)
  PASS: schedule-approvals: no empty catches (0 try/catch, 0 promise)
  PASS: schedule-calendar: no empty catches (0 try/catch, 0 promise)
  PASS: schedule-cloud: no empty catches (0 try/catch, 0 promise)
  PASS: schedule-core: no empty catches (0 try/catch, 0 promise)
  PASS: schedule-evidence: no empty catches (0 try/catch, 0 promise)
  PASS: schedule-old: no empty catches (0 try/catch, 0 promise)
  PASS: setup-wizard: no empty catches (0 try/catch, 0 promise)
  PASS: solace: no empty catches (0 try/catch, 0 promise)
  PASS: theme: no empty catches (0 try/catch, 0 promise)
  PASS: yinyang-delight: no empty catches (0 try/catch, 0 promise)
  PASS: yinyang-oauth3-confirm: no empty catches (0 try/catch, 0 promise)
  PASS: yinyang-rail: no empty catches (0 try/catch, 0 promise)
  PASS: yinyang-tutorial-v2: no empty catches (0 

In [6]:
# PROBE 5: No global variable pollution — functions should be scoped
# Tower ref: T1 F2 (Kent Beck), T16 F3 (AI Coders — module pattern)
print("=" * 60)
print("PROBE 5: Global Scope / Module Pattern")
print("=" * 60)

for jsf in JS_FILES:
    src = jsf.read_text(encoding="utf-8")

    # Check for IIFE or DOMContentLoaded wrapper (module pattern)
    has_iife = bool(re.search(r'\(function\s*\(', src))
    has_dcl = "DOMContentLoaded" in src or "addEventListener" in src
    has_strict = "'use strict'" in src or '"use strict"' in src

    # Check for unscoped variable assignments at top level
    # (simplified: lines at indent=0 that assign to window.xxx or just xxx = )
    top_level_assigns = []
    for i, line in enumerate(src.split("\n"), 1):
        if not line or line[0] in (' ', '\t', '/', '*', '}', ')'):
            continue
        stripped = line.strip()
        if stripped.startswith("//") or stripped.startswith("/*"):
            continue
        # window.xxx = ... is intentional global export (OK but note it)
        if "window." in stripped and "=" in stripped:
            top_level_assigns.append((i, "window export"))
        # bare function declarations are OK
        elif stripped.startswith("function ") or stripped.startswith("const ") or stripped.startswith("let "):
            continue
        # Top-level bare assignment without const/let/var
        elif re.match(r'^[a-zA-Z_]\w*\s*=', stripped):
            top_level_assigns.append((i, "bare assignment"))

    record(f"P5-scope-{jsf.stem}",
           f"{jsf.stem}: properly scoped ({len(top_level_assigns)} top-level assigns)",
           len(top_level_assigns) <= 3,
           f"top-level: {[(l, t) for l, t in top_level_assigns[:5]]}" if len(top_level_assigns) > 3 else "",
           "T1-F2")

    # Document.write is forbidden
    record(f"P5-no-docwrite-{jsf.stem}",
           f"{jsf.stem}: no document.write()",
           "document.write" not in src,
           tower_ref="T6-F1")

PROBE 5: Global Scope / Module Pattern
  PASS: layout: properly scoped (0 top-level assigns)
  PASS: layout: no document.write()
  PASS: onboarding: properly scoped (0 top-level assigns)
  PASS: onboarding: no document.write()
  PASS: schedule-approvals: properly scoped (0 top-level assigns)
  PASS: schedule-approvals: no document.write()
  PASS: schedule-calendar: properly scoped (0 top-level assigns)
  PASS: schedule-calendar: no document.write()
  PASS: schedule-cloud: properly scoped (0 top-level assigns)
  PASS: schedule-cloud: no document.write()
  PASS: schedule-core: properly scoped (0 top-level assigns)
  PASS: schedule-core: no document.write()
  PASS: schedule-evidence: properly scoped (0 top-level assigns)
  PASS: schedule-evidence: no document.write()
  PASS: schedule-old: properly scoped (0 top-level assigns)
  PASS: schedule-old: no document.write()
  PASS: setup-wizard: properly scoped (0 top-level assigns)
  PASS: setup-wizard: no document.write()
  PASS: solace: prope

In [7]:
# EVIDENCE SUMMARY — Convergence check
print("=" * 60)
print("EVIDENCE SUMMARY")
print("=" * 60)

total = len(results)
passed = sum(1 for r in results if r["status"] == "PASS")
failed = sum(1 for r in results if r["status"] == "FAIL")
score = (passed / total * 100) if total else 0
converged = score >= MIN_SCORE

print(f"\nTotal probes:   {total}")
print(f"Passed:         {passed}")
print(f"Failed:         {failed}")
print(f"Score:          {score:.1f}%")
print(f"MIN_SCORE:      {MIN_SCORE}%")
print(f"Converged:      {'YES' if converged else 'NO'}")

if failed > 0:
    print(f"\n{'='*60}")
    print("FAILURES:")
    print(f"{'='*60}")
    for r in results:
        if r["status"] == "FAIL":
            print(f"  FAIL: {r['name']}" + (f" -- {r['detail']}" if r['detail'] else ""))

evidence = json.dumps(results, sort_keys=True)
evidence_hash = hashlib.sha256(evidence.encode()).hexdigest()[:16]
print(f"\nEvidence hash:  {evidence_hash}")
print(f"Timestamp:      {datetime.now().isoformat()}")
print(f"Notebook:       {NOTEBOOK_PATH}")

summary = {
    "northstar": NORTHSTAR,
    "notebook": NOTEBOOK_PATH,
    "total": total,
    "passed": passed,
    "failed": failed,
    "score": round(score, 1),
    "converged": converged,
    "evidence_hash": evidence_hash,
    "findings": [r for r in results if r["status"] == "FAIL"]
}

EVIDENCE SUMMARY

Total probes:   112
Passed:         112
Failed:         0
Score:          100.0%
MIN_SCORE:      70%
Converged:      YES

Evidence hash:  643ad94ea2088d0c
Timestamp:      2026-03-06T10:24:37.122187
Notebook:       notebooks/qa/02-javascript-qa.ipynb
